# July has 400 eruptions and December has 212. Is that the Earth, or the catalogue?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2FT3_july_or_the_catalogue.ipynb).

Count the eruptions the Smithsonian has recorded since 1950 and sort them by the month they
began. Eleven of the months hold between 212 and 275; July holds 400.

That is a big number to explain. Volcanoes have been argued to erupt seasonally — winter snow and
summer meltwater load and unload the crust, sea level breathes a few millimetres a year with the
monsoon, and both change the stress on a magma chamber by a little. If that were the cause, July
would be a real fact about the Earth.

There is a second explanation, and it is not about volcanoes at all. A catalogue is written by
people, over two centuries, from ships' logs and newspaper reports and satellite passes, and what
those people did when they were unsure is itself recorded in the file. This project is about
telling the two apart — and then about what is left over once you have.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## How this notebook is different

This is a **project track**. It is not a weekly notebook and it does not behave like one.

A weekly notebook shows you a move, walks you through it, and then asks you to make it once
yourself. This one loads the data and reproduces the single result its title names — the July
peak — and then stops helping. From there on every section is a sentence describing what to find
out and an empty cell to find it out in. There is no worked example above to pattern-match
against, because on a real question there never is one.

**There is exactly one self-check in this notebook, and it is on the data loading.** After that,
nothing tells you whether you are right. That is not an oversight and it is not laziness: past the
loading step there is no single right answer here, so a cell that said `assert` would be lying to
you about how research works. What replaces it is the thing researchers actually use — a result
you can get two ways, a number you can predict before you compute it, and a claim you can try to
break.

**And it does not close.** The last section is a question this course does not know the answer to.
Everything above it is scaffolding; that question is the project.

## What you'll be able to do

**The science.** Say whether a seasonal signal in a global eruption catalogue is a fact about the
Earth or a fact about the record, and defend the answer with a number rather than an adjective.
Then say what would have to be true for the question to be answerable at all.

**The skills.** Read a real catalogue's missing-value conventions, including the ones that do not
look missing. Build a null by simulation instead of looking up a table. Put an interval on a test
statistic by resampling the thing that is actually independent, which is rarely the row.

## Setup

The Global Volcanism Program publishes its whole Holocene eruption table as one CSV, with no key
and no login. The cell below reads it live and falls back to the copy stored with the course.

**Read this before you go on — it is the whole project.** Three columns record when an eruption
began, and each has its own way of saying *we do not know*:

- `StartDateYear` is always filled in.
- `StartDateMonth` and `StartDateDay` use **`0` for unknown**, not a blank. A `0` survives
  `dropna()` and then behaves like a number.
- `StartDateDayUncertainty` is the number of days the recorded date could be wrong by. When the
  compilers knew only the year, they still wrote a full date — and put the uncertainty here.

That third column is the one nobody reads.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(url, cache_name):
    """Read the live catalogue; fall back to the copy stored with the course."""
    try:
        return pd.read_csv(url)
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + cache_name)

GVP = ("https://webservices.volcano.si.edu/geoserver/GVP-VOTW/ows?service=WFS&version=1.0.0"
       "&request=GetFeature&typeName=GVP-VOTW:Smithsonian_VOTW_Holocene_Eruptions"
       "&outputFormat=csv")

eruptions = load(GVP, "trackT3_gvp_eruptions.csv")
print("the whole Holocene table:", eruptions.shape)
print(eruptions[["Volcano_Name", "StartDateYear", "StartDateMonth", "StartDateDay",
                 "StartDateDayUncertainty"]].head())

## The result you are handed

Two lines of filtering, and then the count. `StartDateMonth > 0` is doing real work: across the
whole table 3,818 eruptions carry month `0` and another 250 carry a
blank, and `0` is not a blank.

The Smithsonian edits this catalogue continuously, so your counts may differ from the ones printed
in this notebook by a few. Say so if they do — a record that changes under you is the subject here,
not a nuisance.

In [ ]:
dated = eruptions[(eruptions["StartDateYear"] >= 1950) & (eruptions["StartDateMonth"] > 0)]
months = dated["StartDateMonth"]

print("eruptions in the whole table: ", len(eruptions))
print("since 1950, carrying a month:", len(dated))

In [ ]:
assert "StartDateDayUncertainty" in eruptions.columns, \
    "the uncertainty column is missing — the catalogue was read wrong, or its schema changed"
assert 2900 < len(dated) < 3100, \
    "expected about 2977 eruptions; a much larger number means the 1950 cut is missing"
print(f"✓ the data — {len(eruptions)} eruption records, {len(dated)} of them since 1950 "
      f"with a real month")

### And that is the last self-check in this notebook

The pipeline is now trustworthy: the file is the file, the filter is the filter, the counts below
are the counts. Everything from here is yours, and nothing will tell you when you have it right.

## The count that started this

One bar per month. This is the entire observation the project exists to explain, and it needs no
statistics to see.

In [ ]:
per_month = months.value_counts().reindex(range(1, 13)).fillna(0)

plt.bar(per_month.index, per_month.values, color="0.4")
plt.axhline(len(dated) / 12, color="firebrick", lw=1.2)
plt.xlabel("month the eruption began (1 = January)")
plt.ylabel("eruptions")
plt.title(f"Eruptions by month since 1950 (n = {len(dated)}); the line is an even spread")
plt.locator_params(axis="x", integer=True)
plt.show()

print(per_month.astype(int).to_dict())

Eleven bars sit near the line. One does not. To say *how far* from the line the whole picture is,
in one number, add up the squared miss of each bar in units of the bar's own expected height — the
chi-squared statistic. The function below is the only piece of machinery this notebook hands you,
and every section after it uses it again.

Then the number needs something to be compared against, and rather than look one up we make it.
**Monte Carlo:** Make up a world where the effect is absent, a thousand times, and see how often chance alone beats what you measured.

In [ ]:
def chi_squared(month_values):
    """How far a set of months is from twelve equal piles, in the usual chi-squared units."""
    expected = len(month_values) / 12
    total = 0
    for m in range(1, 13):
        observed = (month_values == m).sum()
        total = total + (observed - expected) ** 2 / expected
    return total


def null_spread(n, runs=20000):
    """The chi-squared that a world with no seasonality at all produces, `runs` times over."""
    rng = np.random.default_rng(88)
    out = []
    for i in range(runs):
        out.append(chi_squared(rng.integers(1, 13, size=n)))
    return np.array(out)

In [ ]:
observed = chi_squared(months)
no_season = null_spread(len(dated))

print("chi-squared of the real months:", round(observed, 2))
print("no-season worlds that reached it:", (no_season >= observed).sum(), "out of", len(no_season))
print("95th percentile of the no-season worlds:", round(np.percentile(no_season, 95), 2))

So the July peak is not a small-numbers accident: **120.2** against a no-season world
that reaches only **19.6** nineteen times in twenty. A hypothesis test would
stop here and report seasonality.

### Predict before you run

The July bar stands **152** eruptions above the even line. Of those 152, how
many do you think are eruptions that really began in July? Change `my_guess` and run the cell —
you will check it two sections from now, and a wrong guess you committed to is worth more than a
right answer you were shown.

In [ ]:
my_guess = 76

print("I think", my_guess, "of the 152 extra July eruptions really began in July")

## Where the dates come from

A month is made of days, and the days are in the file too.

### ✏️ Your turn 1

Count the 2,977 eruptions by **day of month** — 1 to 31, ignoring which month — and draw
them as a bar chart. Then say, in the output, which days are not like the others and by how much.

You are looking for structure that has nothing to do with volcanoes. Print the median day-count
alongside the two largest so the comparison is on the page and not in your head.

In [ ]:
# ← your answer here



The typical day of the month holds 81 eruptions. Day 15 —
the largest of the ordinary days — holds 115. Day 2 holds 217 and day 16 holds
390.

Nothing about volcanoes distinguishes the 16th of a month from the 15th. Somebody wrote those
dates down.

### ✏️ Your turn 2

`StartDateDayUncertainty` is the column the setup cell told you nobody reads. Use it to find out
what day 2 and day 16 are.

Three things to put on the page:

1. What uncertainty values the day-2 and day-16 records carry — `value_counts()` on that column,
   for each of those two days.
2. Of the **400** July eruptions, how many fall on 2 July, and how many of those carry an
   uncertainty large enough to cover half a year.
3. The number that matters: the July bar stands 152 above the even line, so what
   **fraction of that excess** is accounted for by the one date you have just found? Print it.

Compare the fraction with the number you wrote down in *Predict before you run*.

In [ ]:
# ← your answer here



Every one of the 134 records in this window carrying an uncertainty of exactly 182 days
is dated **2 July** — the middle of the year. When the compilers knew the year and nothing else,
they wrote the midpoint of the year and recorded, in a column that never appears on a plot, that
the date was worth nothing. Day 16 is the same move one level down: 311 of the
316 records carrying an uncertainty of 15 days sit on the 16th, the middle of a month.

134 of the 152-eruption July excess — **88%** —
is that one placeholder.

## Which records do you trust

You now have to throw some data away, and there is no correct amount. Three cuts are defensible:

- **keep everything** — 2,977 eruptions, the analysis you have already done;
- **drop the placeholders** — anything with `StartDateDayUncertainty >= 15`, which removes
  both the year-level and the month-level defaults;
- **keep only exact dates** — rows where the uncertainty is blank, meaning the compilers claimed
  the day itself.

Each is a different answer to a different question, and they do not agree. This is the one real
decision in this track; make it, and report what it cost.

### ✏️ Your turn 3

Build all three subsets. For each one print: how many eruptions it holds, its chi-squared, the 95th
percentile of `null_spread` **for that sample size** (the threshold moves with n, so it has to be
recomputed), and which month is the tallest bar.

Then draw the month bar chart for the middle cut beside the one from *The result you are handed*,
so the before and after are on the same page.

Watch the blank-versus-zero distinction one more time: an exact date has a **blank** uncertainty,
so `.isna()` is what selects it, and a comparison like `>= 15` is False for a blank rather
than True.

In [ ]:
# ← your answer here



Dropping the placeholders removes 497 of the 2,977 records and takes the
statistic from **120.2** to **20.9**, against a no-season
95th percentile of **19.5**. July falls from
1.61× the even height to 1.09×, and the tallest
bar is now March at 244.

Keep only the exact dates and the 2,357 that remain give
**18.1** against a 95th percentile of
19.6 — below it. Same catalogue, same question, and the answer flips
on a choice about which rows to believe.

### ✏️ Your turn 4

Two or three paragraphs, quoting **your own three chi-squared values and their three thresholds**.

1. Which cut would you report, and what does it cost you? Say what the discarded rows were and
   what a reader loses by not seeing them.
2. The middle cut lands just above its threshold and the strict cut lands just below. Name what a
   reader should conclude from a result that changes side when you change a defensible choice —
   and say what would have to be true of the data for the two cuts to agree.

*(Double-click this cell and replace this line with your answer.)*

## Counting what is independent

A chi-squared test asks how surprising a set of counts is if every observation were an independent
draw. That assumption is not about the arithmetic; it is about the world. Before quoting
20.9 against 19.5, it is worth asking how
many independent things the 2,480 rows really are.

### ✏️ Your turn 5

For the middle cut, count the **volcanoes**, not the eruptions. `Volcano_Number` identifies one.

Print: how many distinct volcanoes there are, the mean and median number of eruptions per volcano,
how many volcanoes contribute exactly one, and the name and count of the two busiest. Draw
whatever figure makes the shape of that distribution obvious.

In [ ]:
# ← your answer here



2,480 eruptions, **364 volcanoes**. The median volcano
contributes 3; Fournaise, Piton de la contributes 64 and
Bezymianny another 52. One volcano's eruptive episode is one thing
happening, not 64 independent draws from the calendar — and a chi-squared test that
counts it as 64 is counting the same evidence over and over.

The fix is the same move the uncertainty week made, applied to the right unit.
**Bootstrap:** Ask the data the same question a thousand times, using a different random slice of itself each time.

### ✏️ Your turn 6

Bootstrap the chi-squared **by volcano**, not by eruption. The recipe, in words:

1. Split the middle cut into one group per `Volcano_Number` and keep each group's months. A
   `for volcano_id, rows in clean.groupby("Volcano_Number"):` loop gives you the groups one at a
   time; collect `rows["StartDateMonth"].values` into a list, one entry per volcano.
2. 364 times over, and 2000 times in all: draw 364 volcanoes
   **with replacement** — `rng.integers(0, n_volcanoes, size=n_volcanoes)` gives you their
   positions — glue their month arrays together with `np.concatenate(parts)`, and take
   `chi_squared` of the result.
3. Report the 2.5th and 97.5th percentiles of the 2000 statistics, the median, and the fraction
   of them that fall **below** the no-season threshold you computed in Your turn 3.

Draw the 2000 statistics as a histogram with the observed value and the threshold marked.

**Confidence interval:** Not one number but the range your number would have wandered over, had the world rolled differently.

In [ ]:
# ← your answer here



Resampling volcanoes rather than eruptions spreads the statistic from
**14.2 to 57.0**, median 31.2, and
**10%** of the resamples land below the threshold the observed value
cleared. A statistic whose interval is that wide has not established anything; the margin of
1.4 it cleared by is inside its own noise.

The naive chi-squared is not the honest test on this data, and the reason is not the arithmetic —
it is that the rows are not the independent units the test assumes.

## The question, answered

July has 400 eruptions and December has 212 because the Smithsonian
writes **2 July** when it knows only the year: 134 such records, about
88% of the July excess. That answers the title. It also removes the
question — cleaned of placeholders, this catalogue does not show a seasonal signal that survives a
test which respects how few independent volcanoes it contains.

## What track T3 leans on

**The question.** July has 400 eruptions and December has 212. Is that the Earth, or the catalogue?

Nothing here is new. These are the weeks to look back at while you work, and the wording is the course's own.

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Monte Carlo** | Make up a world where the effect is absent, a thousand times, and see how often chance alone beats what you measured. |
| **Bootstrap** | Ask the data the same question a thousand times, using a different random slice of itself each time. |
| **Confidence interval** | Not one number but the range your number would have wandered over, had the world rolled differently. |
| **Mask** | Comparing an array with a number asks the same question of every cell at once and hands back a grid of True and False. |
| **Table** | A table with a name on every column, so you ask for data by name instead of by position. |

### Code you will reach back for

| Function | What it does |
|---|---|
| `table.sort_values(by)` | put the rows in order by one column |
| `column.value_counts()` | how often each value appears |
| `table.groupby(column)` | split the table into one group per value |
| `column.isna()` | a mask marking where the file had nothing |
| `np.random.default_rng(seed) / rng.shuffle(a)` | a random-number generator you can reproduce, and a repeatable shuffle |
| `rng.integers(low, high, size)` | that many whole numbers drawn at random |
| `np.percentile(values, 95)` | the value that 95 per cent of the numbers fall below |
| `np.percentile(values, [2.5, 97.5])` | the two values that cut off the bottom and top 2.5% — a 95% interval |

## What your project must contain

Five sections, empty below, required of **every** EPS 88 project regardless of track. They are
headed here so the shape of a good answer is visible while you work. Fill them in as you go; they
are not a write-up you do at the end.

### ✏️ 1 · A one-sentence answer

Your claim and its uncertainty, in one sentence, at the top of your report. If you cannot put a
number and a range in it, you do not have a result yet.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 2 · The trivial baseline

Before any statistic, state the dumbest answer to your question and what it gives. Every later
number is reported against it.

On this track the honest baseline is not a model at all — it is a bar chart, and it already
answers the title. Say what the simplest possible answer is, what it gives, and what each later
step actually bought you over it.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 3 · Split by structure

Earth data are correlated in space and in time, so whatever you split, resample or count as
independent has to be split along the structure that is really there — never at random across
rows.

This track fits no model, so there is no train/test split to get wrong. The same idea has teeth
anyway, and *Your turn 6* is where it bit. Name the unit you treated as independent, say why, and
say what changed when you got it right.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 4 · What I got wrong

What failed, and what you believed before it failed. Honest failure is graded; a faked success is
not. Your *Predict before you run* guess belongs here if it was wrong.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 5 · AI disclosure

Which tool, what you asked it, what you changed in what it gave you, and how you checked that the
result was true.

*(Double-click this cell and replace this line with your answer.)*

## The open question

> **Could a real seasonal signal ever be detected in a catalogue like this?**

Nobody grading this knows the answer, and neither does the literature. Everything above is the
scaffolding; this is the project.

Here is what is actually established, and it is less than it looks. The July peak is a placeholder
and that is settled. What is **not** settled is the next question down: after the placeholders are
gone, is the 20.9 that remains a small real signal that this catalogue
is too coarse to resolve, or nothing at all? The volcano-block interval —
14.2 to 57.0 — says the data cannot currently tell you.

Three directions, none of them worked out here:

1. **Fix the null.** The test above assumes twelve equal months. They are not equal: February is
   short and seven months have 31 days, so a uniform expectation is wrong by a few percent before
   any physics. Recompute the expected counts from the real lengths of the months and see which
   way — and how far — the statistic moves.
2. **Change the unit, not the test.** If a volcano is the independent thing, one obvious move is to
   ask each volcano a single question — its own peak month, say, or a per-volcano statistic — and
   test the 364 answers rather than the 2,480 eruptions. What
   power would such a test have, and is 364 enough?
3. **Split the Earth in half.** July is midsummer north of the equator and midwinter south of it.
   Any mechanism that runs on snow load, meltwater or the seasonal sea-level cycle must therefore
   push the two hemispheres in *opposite* directions, while a recording artefact pushes them the
   same way. `GeoLocation` holds a `POINT (lon lat)` string for every row, so this is a filter and
   two bar charts. It is the sharpest test available in this dataset, and this notebook has
   deliberately not run it.

And one that is bigger than a semester: what would a catalogue have to look like for a seasonal
signal of a few percent to be **detectable** at all? Count what you would need — how many
independent volcanoes, over how long, with dates good to what precision — and compare it with what
364 volcanoes and 2,480 eruptions can support. If the answer
is that no achievable catalogue could settle it, that is a result, and it is the one this project
is most likely to reach.

### ✏️ Your turn 7 — the first move

Before you close this notebook: in a few sentences, name the **one** measurement you would make
first, say what it would show if the seasonal signal is real, what it would show if it is not, and
name the number that would change your mind. Then make it, in the cell below the prose.

*(Double-click this cell and replace this line with your answer.)*

In [ ]:
# ← your answer here

